In [1]:
import pickle
import torch

# Load and check what we have
scalers = pickle.load(open('research\generation_output\gru_v2_final_scalers.pkl', 'rb'))
states = torch.load('research\generation_output\gru_v2_final_models.pt', map_location='cpu')

print("=== SAVED SCALERS ===")
for key in sorted(scalers.keys()):
    s = scalers[key]
    print(f"  {key}: {s.n_features_in_} features")

print("\n=== SAVED MODEL STATES ===")
for name, state in states.items():
    print(f"\n  {name}:")
    for k, v in state.items():
        print(f"    {k}: {v.shape}")

<>:5: SyntaxWarning: invalid escape sequence '\g'
<>:6: SyntaxWarning: invalid escape sequence '\g'
<>:5: SyntaxWarning: invalid escape sequence '\g'
<>:6: SyntaxWarning: invalid escape sequence '\g'
C:\Users\Aditya\AppData\Local\Temp\ipykernel_14648\3153029615.py:5: SyntaxWarning: invalid escape sequence '\g'
  scalers = pickle.load(open('research\generation_output\gru_v2_final_scalers.pkl', 'rb'))
C:\Users\Aditya\AppData\Local\Temp\ipykernel_14648\3153029615.py:6: SyntaxWarning: invalid escape sequence '\g'
  states = torch.load('research\generation_output\gru_v2_final_models.pt', map_location='cpu')


=== SAVED SCALERS ===
  Level_0a_GPS_input: 12 features
  Level_0a_GPS_target: 2 features
  Level_0b_Mobility_input: 16 features
  Level_0b_Mobility_target: 4 features
  Level_0c_Weather_input: 15 features
  Level_0c_Weather_target: 5 features
  Level_0d_Traffic_input: 14 features
  Level_0d_Traffic_target: 2 features
  Level_1_Signal_input: 29 features
  Level_1_Signal_target: 6 features
  Level_2_CellConfig_input: 33 features
  Level_2_CellConfig_target: 4 features
  Level_3a_Downlink_input: 34 features
  Level_3a_Downlink_target: 6 features
  Level_3b_Uplink_input: 31 features
  Level_3b_Uplink_target: 3 features
  Level_4_QoS_input: 41 features
  Level_4_QoS_target: 4 features
  Level_5_Ping_input: 42 features
  Level_5_Ping_target: 1 features

=== SAVED MODEL STATES ===

  Level_0a_GPS:
    gru.weight_ih_l0: torch.Size([384, 12])
    gru.weight_hh_l0: torch.Size([384, 128])
    gru.bias_ih_l0: torch.Size([384])
    gru.bias_hh_l0: torch.Size([384])
    gru.weight_ih_l1: torch.Size

c:\Users\Aditya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
import os
import sys
import numpy as np
import pandas as pd
import sumolib

# ============================================================
# SUMO VALIDATION MODULE
# ============================================================

sumo_home = os.path.expanduser('~/sumo')
sys.path.append(os.path.join(sumo_home, 'tools'))
os.environ['SUMO_HOME'] = sumo_home

SUMO_DIR = 'D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1'
net_file = os.path.join(SUMO_DIR, 'osm1.net.xml')

print("Loading SUMO network...")
net = sumolib.net.readNet(net_file)
print(f"  Network loaded: {len(net.getEdges())} edges")

<>:15: SyntaxWarning: invalid escape sequence '\T'
<>:15: SyntaxWarning: invalid escape sequence '\T'
C:\Users\Aditya\AppData\Local\Temp\ipykernel_14648\816063891.py:15: SyntaxWarning: invalid escape sequence '\T'
  SUMO_DIR = 'D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1'


Loading SUMO network...
  Network loaded: 6055 edges


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import os

# ============================================================
# GRU CAUSAL CHAIN — GENERATION PIPELINE
# ============================================================

device_torch = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LEN = 20
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2

In [4]:
# ============================================================
# 1. MODEL DEFINITION
# ============================================================
class CausalGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim,
                          num_layers=num_layers,
                          dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

In [5]:
# ============================================================
# 2. CHAIN CONFIGURATION
# ============================================================
BASE_INPUTS = [
    'hour', 'day_of_week',
    'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4',
    'direction_uplink',
    'measured_qos_delay',
    'measurement', 'operator'
]

CAUSAL_CHAIN = [
    {
        'name': 'Level_0a_GPS',
        'targets': ['Latitude', 'Longitude'],
        'extra_inputs': [],
        'auto_inputs': ['Latitude', 'Longitude'],
    },
    {
        'name': 'Level_0b_Mobility',
        'targets': ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude'],
        'extra_inputs': ['Latitude', 'Longitude'],
        'auto_inputs': ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude'],
    },
    {
        'name': 'Level_0c_Weather',
        'targets': ['precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed'],
        'extra_inputs': [],
        'auto_inputs': ['precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed'],
    },
    {
        'name': 'Level_0d_Traffic',
        'targets': ['Traffic Jam Factor', 'Traffic Distance'],
        'extra_inputs': ['Latitude', 'Longitude'],
        'auto_inputs': ['Traffic Jam Factor', 'Traffic Distance'],
    },
    {
        'name': 'Level_1_Signal',
        'targets': ['PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                     'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'precipIntensity', 'precipProbability', 'temperature',
                         'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance'],
        'auto_inputs': ['PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                        'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
    },
    {
        'name': 'Level_2_CellConfig',
        'targets': ['PCell_Downlink_frequency', 'PCell_Band_Indicator',
                     'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'precipIntensity', 'precipProbability', 'temperature',
                         'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
        'auto_inputs': ['PCell_Downlink_frequency', 'PCell_Band_Indicator',
                        'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
    },
    {
        'name': 'Level_3a_Downlink',
        'targets': ['PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                     'PCell_Downlink_TB_Size',
                     'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'auto_inputs': ['PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                        'PCell_Downlink_TB_Size',
                        'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High'],
    },
    {
        'name': 'Level_3b_Uplink',
        'targets': ['PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                     'PCell_Uplink_Tx_Power_(dBm)'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'auto_inputs': ['PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                        'PCell_Uplink_Tx_Power_(dBm)'],
    },
    {
        'name': 'Level_4_QoS',
        'targets': ['datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                         'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                         'PCell_Downlink_TB_Size',
                         'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
                         'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                         'PCell_Uplink_Tx_Power_(dBm)'],
        'auto_inputs': ['datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate'],
    },
    {
        'name': 'Level_5_Ping',
        'targets': ['ping_ms'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                         'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                         'PCell_Downlink_TB_Size',
                         'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
                         'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                         'PCell_Uplink_Tx_Power_(dBm)',
                         'datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate'],
        'auto_inputs': ['ping_ms'],
    },
]

# Snapping rules for discrete features
SNAP_RULES = {
    'PCell_Downlink_frequency': [125.0, 475.0, 1300.0, 1801.0, 2850.0, 3050.0, 3749.0, 9460.0],
    'PCell_freq_MHz': [700.0, 900.0, 1800.0, 2000.0, 2100.0, 2600.0],
    'PCell_Band_Indicator': [1.0, 3.0, 7.0, 8.0, 28.0],
    'PCell_Downlink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Uplink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Downlink_Average_MCS': list(range(0, 30)),
}

def snap_to_nearest(value, valid_set):
    valid_arr = np.array(valid_set)
    return valid_arr[np.argmin(np.abs(valid_arr - value))]

# All generated columns (in model space)
ALL_GENERATED_COLS = []
for level in CAUSAL_CHAIN:
    ALL_GENERATED_COLS.extend(level['targets'])

In [6]:
# ============================================================
# 3. LOAD MODELS AND SCALERS
# ============================================================
MODEL_DIR = 'research\generation_output'

print("Loading models and scalers...")
saved_scalers = pickle.load(open(os.path.join(MODEL_DIR, 'gru_v2_final_scalers.pkl'), 'rb'))
saved_states = torch.load(os.path.join(MODEL_DIR, 'gru_v2_final_models.pt'), map_location=device_torch)

# Rebuild models
chain_models = {}
chain_scalers = {}

for level in CAUSAL_CHAIN:
    name = level['name']
    context_cols = BASE_INPUTS + level['extra_inputs']
    auto_cols = level['auto_inputs']
    n_input = len(context_cols) + len(auto_cols)
    n_output = len(level['targets'])

    model = CausalGRU(n_input, HIDDEN_SIZE, n_output, NUM_LAYERS, DROPOUT).to(device_torch)
    model.load_state_dict(saved_states[name])
    model.eval()

    chain_models[name] = model
    chain_scalers[f"{name}_input"] = saved_scalers[f"{name}_input"]
    chain_scalers[f"{name}_target"] = saved_scalers[f"{name}_target"]

    print(f"  Loaded {name} (in={n_input}, out={n_output})")

# Load dataset for seed windows
print("\nLoading dataset...")
df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')
df = df.sort_values('timestamp').reset_index(drop=True)
df['sin_COG'] = np.sin(np.radians(df['COG']))
df['cos_COG'] = np.cos(np.radians(df['COG']))
df['jitter_log'] = np.log1p(df['jitter'])
print(f"Dataset loaded: {len(df)} rows")

<>:4: SyntaxWarning: invalid escape sequence '\g'
<>:33: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\g'
<>:33: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Aditya\AppData\Local\Temp\ipykernel_14648\486828071.py:4: SyntaxWarning: invalid escape sequence '\g'
  MODEL_DIR = 'research\generation_output'
C:\Users\Aditya\AppData\Local\Temp\ipykernel_14648\486828071.py:33: SyntaxWarning: invalid escape sequence '\m'
  df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')


Loading models and scalers...
  Loaded Level_0a_GPS (in=12, out=2)
  Loaded Level_0b_Mobility (in=16, out=4)
  Loaded Level_0c_Weather (in=15, out=5)
  Loaded Level_0d_Traffic (in=14, out=2)
  Loaded Level_1_Signal (in=29, out=6)
  Loaded Level_2_CellConfig (in=33, out=4)
  Loaded Level_3a_Downlink (in=34, out=6)
  Loaded Level_3b_Uplink (in=31, out=3)
  Loaded Level_4_QoS (in=41, out=4)
  Loaded Level_5_Ping (in=42, out=1)

Loading dataset...
Dataset loaded: 204942 rows


In [7]:
# ============================================================
# 1. ROAD POSITION CALCULATOR
# ============================================================
def get_road_position(timestamp, last_row, net=net):
    """
    Given a new timestamp, calculate where the car would be
    on the road network if it continued driving.
    
    Parameters:
        timestamp:  new timestamp (str or pd.Timestamp)
        last_row:   last real data row (needs Latitude, Longitude, speed_kmh, COG, timestamp)
        net:        sumolib network
    
    Returns:
        dict with lat, lon, edge name, speed_limit, distance, time_diff
    """
    ts = pd.Timestamp(timestamp)
    if ts.tzinfo is None:
        ts = ts.tz_localize('Europe/Berlin')
    
    last_ts = pd.Timestamp(last_row['timestamp'])
    time_diff = (ts - last_ts).total_seconds()
    
    if time_diff <= 0:
        return None
    
    speed_ms = max(last_row['speed_kmh'] / 3.6, 1.0)
    target_dist = speed_ms * time_diff
    last_cog = last_row['COG']
    
    # Find current edge and offset
    x, y = net.convertLonLat2XY(last_row['Longitude'], last_row['Latitude'])
    nearby = net.getNeighboringEdges(x, y, r=100)
    valid = [(e, d) for e, d in nearby if not e.getID().startswith(':')]
    if not valid:
        valid = nearby
    if not valid:
        return None
    
    edge, _ = sorted(valid, key=lambda e: e[1])[0]
    
    # Find offset on edge
    shape = edge.getShape()
    min_d = float('inf')
    car_offset = 0
    for i in range(len(shape) - 1):
        x1, y1 = shape[i]
        x2, y2 = shape[i + 1]
        dx, dy = x2 - x1, y2 - y1
        seg_len = np.sqrt(dx**2 + dy**2)
        if seg_len > 0:
            t = max(0, min(1, ((x - x1)*dx + (y - y1)*dy) / seg_len**2))
            px, py = x1 + t*dx, y1 + t*dy
            d = np.sqrt((x - px)**2 + (y - py)**2)
            if d < min_d:
                min_d = d
                car_offset = sum(
                    np.sqrt((shape[k+1][0]-shape[k][0])**2 + (shape[k+1][1]-shape[k][1])**2)
                    for k in range(i)
                ) + t * seg_len
    
    # Walk along road
    dist_remaining = target_dist
    offset = car_offset
    current_edge = edge
    
    while dist_remaining > 0:
        remaining_on_edge = current_edge.getLength() - offset
        
        if dist_remaining <= remaining_on_edge:
            final_offset = offset + dist_remaining
            pos = sumolib.geomhelper.positionAtShapeOffset(current_edge.getShape(), final_offset)
            lon, lat = net.convertXY2LonLat(pos[0], pos[1])
            return {
                'lat': lat, 'lon': lon,
                'edge': current_edge.getName() or current_edge.getID(),
                'speed_limit': current_edge.getSpeed() * 3.6,
                'dist_from_car': round(target_dist, 1),
                'time_diff': time_diff,
                'timestamp': str(ts),
            }
        
        dist_remaining -= remaining_on_edge
        offset = 0
        
        outgoing = current_edge.getOutgoing()
        valid_out = [e for e in outgoing if not e.getID().startswith(':')]
        if not valid_out:
            valid_out = list(outgoing) if outgoing else []
        
        if not valid_out:
            pos = sumolib.geomhelper.positionAtShapeOffset(
                current_edge.getShape(), current_edge.getLength())
            lon, lat = net.convertXY2LonLat(pos[0], pos[1])
            return {
                'lat': lat, 'lon': lon,
                'edge': current_edge.getName() or current_edge.getID(),
                'speed_limit': current_edge.getSpeed() * 3.6,
                'dist_from_car': round(target_dist, 1),
                'time_diff': time_diff,
                'timestamp': str(ts),
            }
        
        best_edge = None
        best_diff = 999
        for e in valid_out:
            e_shape = e.getShape()
            if len(e_shape) >= 2:
                heading = np.degrees(np.arctan2(
                    e_shape[1][0] - e_shape[0][0],
                    e_shape[1][1] - e_shape[0][1]
                )) % 360
                diff = abs(last_cog - heading)
                diff = min(diff, 360 - diff)
                if diff < best_diff:
                    best_diff = diff
                    best_edge = e
        
        current_edge = best_edge if best_edge else valid_out[0]
    
    return None


In [8]:
def generate_chain(device_id, start_timestamp, n_steps=1, mode='with_gps',
                   lat=None, lon=None, positions=None):
    """
    Generate LTE telemetry data using GRU causal chain.

    Parameters:
        device_id: str — 'pc1', 'pc2', 'pc3', or 'pc4'
        start_timestamp: str or pd.Timestamp
        n_steps: int — number of timesteps to generate (1-10 recommended)
        mode: str — 'full' or 'with_gps'
        lat: float — latitude (required if mode='with_gps')
        lon: float — longitude (required if mode='with_gps')
        positions: list of dicts with 'lat','lon' per step (overrides lat/lon)

    Returns:
        pd.DataFrame
    """
    start_ts = pd.Timestamp(start_timestamp)
    if start_ts.tzinfo is None:
        start_ts = start_ts.tz_localize('Europe/Berlin')

    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)

    mask = device_data['timestamp'] < start_ts
    if mask.sum() < SEQ_LEN:
        print(f"Error: Need {SEQ_LEN} rows before {start_ts}, found {mask.sum()}")
        return None

    seed_idx = device_data[mask].index[-SEQ_LEN:]
    seed = device_data.loc[seed_idx].reset_index(drop=True)

    # Auto-generate positions along road if single lat/lon given
    if positions is None and mode == 'with_gps' and lat is not None and n_steps > 1:
        last_seed = seed.iloc[-1]
        last_row_fake = {
            'Latitude': lat,
            'Longitude': lon,
            'speed_kmh': last_seed['speed_kmh'],
            'COG': last_seed['COG'],
            'timestamp': start_ts - pd.Timedelta(seconds=1),
        }
        
        positions = [{'lat': lat, 'lon': lon}]
        for sec in range(1, n_steps):
            next_ts = start_ts + pd.Timedelta(seconds=sec)
            road_pos = get_road_position(str(next_ts), last_row_fake, net)
            if road_pos:
                positions.append({'lat': road_pos['lat'], 'lon': road_pos['lon']})
            else:
                positions.append({'lat': lat, 'lon': lon})

    base_buffer = {}
    for col in BASE_INPUTS:
        base_buffer[col] = list(seed[col].values)

    gen_buffer = {}
    for col in ALL_GENERATED_COLS:
        gen_buffer[col] = list(seed[col].values)

    generated_rows = []

    for step in range(n_steps):
        current_ts = start_ts + pd.Timedelta(seconds=step)

        # Get lat/lon for this step
        step_lat = None
        step_lon = None
        if positions is not None and step < len(positions):
            step_lat = positions[step]['lat']
            step_lon = positions[step]['lon']
        elif mode == 'with_gps' and lat is not None:
            step_lat = lat
            step_lon = lon

        current_base = {
            'hour': current_ts.hour,
            'day_of_week': current_ts.dayofweek + 1,
            'device_pc1': 1 if device_id == 'pc1' else 0,
            'device_pc2': 1 if device_id == 'pc2' else 0,
            'device_pc3': 1 if device_id == 'pc3' else 0,
            'device_pc4': 1 if device_id == 'pc4' else 0,
            #'direction_downlink': base_buffer['direction_downlink'][-1],
            'direction_uplink': base_buffer['direction_uplink'][-1],
            #'measured_qos_datarate': base_buffer['measured_qos_datarate'][-1],
            'measured_qos_delay': base_buffer['measured_qos_delay'][-1],
            'measurement': base_buffer['measurement'][-1],
            'operator': base_buffer['operator'][-1],
        }

        for col in BASE_INPUTS:
            base_buffer[col].append(current_base[col])

        for col in ALL_GENERATED_COLS:
            gen_buffer[col].append(0.0)

        for level in CAUSAL_CHAIN:
            level_name = level['name']
            context_cols = BASE_INPUTS + level['extra_inputs']
            auto_cols = level['auto_inputs']
            target_cols = level['targets']

            model_level = chain_models[level_name]
            scaler_in = chain_scalers[f"{level_name}_input"]
            scaler_tgt = chain_scalers[f"{level_name}_target"]

            if mode == 'with_gps' and level_name == 'Level_0a_GPS' and step_lat is not None:
                gen_buffer['Latitude'][-1] = step_lat
                gen_buffer['Longitude'][-1] = step_lon
                continue

            ctx_seq = np.zeros((SEQ_LEN, len(context_cols)), dtype=np.float32)
            auto_seq = np.zeros((SEQ_LEN, len(auto_cols)), dtype=np.float32)

            for t in range(SEQ_LEN):
                buf_idx = len(base_buffer['hour']) - SEQ_LEN + t
                for j, col in enumerate(context_cols):
                    if col in BASE_INPUTS:
                        ctx_seq[t, j] = base_buffer[col][buf_idx]
                    else:
                        ctx_seq[t, j] = gen_buffer[col][buf_idx]
                for j, col in enumerate(auto_cols):
                    auto_seq[t, j] = gen_buffer[col][buf_idx]

            auto_seq[-1, :] = 0.0

            x = np.concatenate([ctx_seq, auto_seq], axis=1)
            x_scaled = scaler_in.transform(x.reshape(-1, x.shape[1])).reshape(1, SEQ_LEN, -1)

            with torch.no_grad():
                pred_scaled = model_level(torch.FloatTensor(x_scaled).to(device_torch)).cpu().numpy()
            pred = scaler_tgt.inverse_transform(pred_scaled)[0]

            for j, col in enumerate(target_cols):
                val = pred[j]
                if col in SNAP_RULES:
                    val = snap_to_nearest(val, SNAP_RULES[col])
                gen_buffer[col][-1] = val

        if step_lat is not None:
            gen_buffer['Latitude'][-1] = step_lat
            gen_buffer['Longitude'][-1] = step_lon

        row = {'timestamp': current_ts}
        for col in BASE_INPUTS:
            row[col] = current_base[col]
        for col in ALL_GENERATED_COLS:
            row[col] = gen_buffer[col][-1]

        row['COG'] = np.degrees(np.arctan2(row['sin_COG'], row['cos_COG'])) % 360
        row['jitter'] = np.expm1(row['jitter_log'])

        generated_rows.append(row)

    return pd.DataFrame(generated_rows)

In [39]:
# ============================================================
# 5. FORMAT OUTPUT (reverse all transforms for display)
# ============================================================
LOG_FEATURES = [
    'datarate', 'target_datarate',
    'PCell_Downlink_TB_Size', 'PCell_Uplink_TB_Size',
    'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
    'PCell_Downlink_Num_RBs', 'PCell_Uplink_Num_RBs',
    'ping_ms', 'Pos in Ref Round', 'Traffic Distance',
    'PCell_Uplink_Tx_Power_(dBm)',
]

def format_output(gen_df):
    """
    Convert model-space output to human-readable format.
    Reverses: log transforms, one-hot encoding, sin/cos COG.
    """
    out = gen_df.copy()

    # Reverse log transforms
    for col in LOG_FEATURES:
        if col in out.columns:
            out[f'{col}'] = np.expm1(out[col])

    # Decode device
    out['device'] = 'unknown'
    for d in ['pc1', 'pc2', 'pc3', 'pc4']:
        mask = out[f'device_{d}'] == 1
        out.loc[mask, 'device'] = d

    # Decode direction
    out['direction'] = 'downlink'
    #out.loc[out['direction_downlink'] == 1, 'direction'] = 'downlink'
    out.loc[out['direction_uplink'] == 1, 'direction'] = 'uplink'

    # Decode measured_qos
    out['measured_qos'] = 'datarate'
    #out.loc[out['measured_qos_datarate'] == 1, 'measured_qos'] = 'datarate'
    out.loc[out['measured_qos_delay'] == 1, 'measured_qos'] = 'delay'

    # Select display columns
    display_cols = [
        'timestamp', 'device', 'direction', 'measured_qos',
        'Latitude', 'Longitude', 'speed_kmh', 'COG', 'Altitude',
        'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed',
        'Traffic Jam Factor',
        'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
        'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
        'PCell_Downlink_frequency', 'PCell_Band_Indicator',
        'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
        'PCell_Downlink_Average_MCS',
        'jitter', 'measurement', 'operator',
    ]

    # Add original-scale log features
    for col in LOG_FEATURES:
        if f'{col}' in out.columns:
            display_cols.append(f'{col}')

    available = [c for c in display_cols if c in out.columns]
    return out[available]


In [40]:
# ============================================================
# 6. COMPARISON FUNCTION
# ============================================================
def compare_with_real(device_id, start_timestamp, n_steps=10, mode='full',
                      lat=None, lon=None):
    """
    Generate and compare against real data.
    """
    start_ts = pd.Timestamp(start_timestamp)
    if start_ts.tzinfo is None:
        start_ts = start_ts.tz_localize('Europe/Berlin')

    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)

    # Get real data
    real_mask = (device_data['timestamp'] >= start_ts) & \
                (device_data['timestamp'] < start_ts + pd.Timedelta(seconds=n_steps))
    real = device_data[real_mask].head(n_steps).reset_index(drop=True)

    # Generate
    gen = generate_chain(device_id, start_timestamp, n_steps, mode, lat, lon)
    if gen is None:
        return None, None

    # Compare key features
    compare_cols = [
        ('Latitude', 'Latitude', ''),
        ('Longitude', 'Longitude', ''),
        ('speed_kmh', 'speed_kmh', 'km/h'),
        ('COG', 'COG', '°'),
        ('Altitude', 'Altitude', 'm'),
        ('temperature', 'temperature', '°C'),
        ('PCell_RSRP_max', 'PCell_RSRP_max', 'dBm'),
        ('PCell_freq_MHz', 'PCell_freq_MHz', 'MHz'),
        ('PCell_Downlink_frequency', 'PCell_Downlink_frequency', ''),
        ('datarate', 'datarate', '(log)'),
        ('ping_ms', 'ping_ms', '(log)'),
        ('jitter_log', 'jitter_log', '(log)'),
    ]

    n = min(len(gen), len(real))

    print(f"\n{'='*100}")
    print(f"COMPARISON: {device_id} | {start_timestamp} | {n_steps} steps | Mode: {mode}")
    print(f"{'='*100}")

    print(f"\n{'Feature':30s} | {'RMSE':>10s} | {'MAE':>10s} | {'Real (first 3)':>30s} | {'Gen (first 3)':>30s}")
    print("─" * 120)

    for real_col, gen_col, unit in compare_cols:
        if real_col not in real.columns or gen_col not in gen.columns:
            continue

        real_vals = real[real_col].values[:n]
        gen_vals = gen[gen_col].values[:n]

        if real_col == 'COG':
            # Circular error
            diff = np.abs(real_vals - gen_vals)
            diff = np.minimum(diff, 360 - diff)
            rmse = np.sqrt(np.mean(diff**2))
            mae = np.mean(diff)
        else:
            rmse = np.sqrt(np.mean((real_vals - gen_vals)**2))
            mae = np.mean(np.abs(real_vals - gen_vals))

        real_str = ', '.join([f'{v:.4f}' for v in real_vals[:3]])
        gen_str = ', '.join([f'{v:.4f}' for v in gen_vals[:3]])

        print(f"{real_col+' '+unit:30s} | {rmse:10.4f} | {mae:10.4f} | {real_str:>30s} | {gen_str:>30s}")

    return gen, real

In [41]:


# ============================================================
# 7. TEST
# ============================================================
print("\n" + "="*60)
print("PIPELINE READY — Running tests")
print("="*60)

# Test 1: Mode 1 — timestamp + device
print("\n--- TEST 1: Mode 'full' (predict everything) ---")
gen1, real1 = compare_with_real(
    device_id='pc1',
    start_timestamp='2021-06-23 14:12:51+02:00',
    n_steps=5,
    mode='with_gps',
    lat=52.504327,
    lon=13.335717
)

# Test 3: Format output for display
print("\n--- TEST 3: Formatted output ---")
if gen1 is not None:
    formatted = format_output(gen1)
    print(f"\nFormatted columns: {list(formatted.columns)}")
    print(formatted.to_string())

print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
print("\nUsage:")
print("  gen = generate_chain('pc1', '2021-06-23 14:12:51+02:00', n_steps=5, mode='with_gps', lat=52.50, lon=13.33)")
print("  formatted = format_output(gen)")
print("  gen, real = compare_with_real('pc1', '2021-06-23 14:12:51+02:00', n_steps=5)")


PIPELINE READY — Running tests

--- TEST 1: Mode 'full' (predict everything) ---


C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")



COMPARISON: pc1 | 2021-06-23 14:12:51+02:00 | 5 steps | Mode: with_gps

Feature                        |       RMSE |        MAE |                 Real (first 3) |                  Gen (first 3)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Latitude                       |     0.0001 |     0.0001 |      52.5043, 52.5043, 52.5043 |      52.5043, 52.5043, 52.5043
Longitude                      |     0.0002 |     0.0001 |      13.3357, 13.3357, 13.3357 |      13.3357, 13.3358, 13.3358
speed_kmh km/h                 |     1.5224 |     1.1656 |         2.7955, 2.7610, 2.6231 |         2.8357, 2.5851, 1.5836
COG °                          |    22.9323 |    20.8759 |   211.6000, 209.4000, 208.8000 |   205.8007, 195.0861, 184.6555
Altitude m                     |     0.6337 |     0.5977 |      39.3000, 39.1000, 38.6000 |      39.7543, 39.7561, 39.5814
temperature °C                 |     0.0203 |     0.0189 |      20.4

In [42]:
# ============================================================
# 2. POINT VALIDATOR
# ============================================================
def validate_point(lat, lon, speed, net=net):
    """
    Validate a single point against the road network.
    
    Returns:
        dict with on_road, snap_dist, speed_ok, speed_limit, road
    """
    x, y = net.convertLonLat2XY(lon, lat)
    nearby = net.getNeighboringEdges(x, y, r=100)
    if not nearby:
        return {'on_road': False, 'snap_dist': 999, 'speed_ok': False,
                'speed_limit': 0, 'road': 'NO_ROAD'}
    valid = [(e, d) for e, d in nearby if not e.getID().startswith(':')]
    if not valid:
        valid = nearby
    nearest, dist = sorted(valid, key=lambda e: e[1])[0]
    sl = nearest.getSpeed() * 3.6
    return {
        'on_road': bool(dist <= 15),
        'snap_dist': round(float(dist), 1),
        'speed_ok': bool(speed <= sl * 1.1 + 5 or speed < 1),
        'speed_limit': round(sl, 1),
        'road': nearest.getName() or nearest.getID(),
    }

In [43]:

# ============================================================
# 3. POSITION ERROR
# ============================================================
def position_error_m(lat1, lon1, lat2, lon2):
    """Haversine distance in meters."""
    R = 6_371_000
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat/2)**2 +
         np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2)
    return round(R * 2 * np.arcsin(np.sqrt(a)), 1)

In [44]:
# ============================================================
# 4. BEYOND-ENDPOINT VALIDATION
# ============================================================
def validate_beyond(segment, generate_fn, model_name, n_beyond=5,
                    device_id='pc1', net=net):
    """
    Validate model at timestamps beyond the end of a segment.
    
    Parameters:
        segment:      DataFrame of a continuous driving segment
        generate_fn:  generation function (generate_direct_lstm or generate_chain)
        model_name:   str for display
        n_beyond:     number of seconds to generate beyond
        device_id:    device ID
    
    Returns:
        dict with results
    """
    last_row = segment.iloc[-1]
    last_ts = pd.Timestamp(last_row['timestamp'])
    
    print(f"\n{'='*80}")
    print(f"  BEYOND-ENDPOINT VALIDATION: {model_name}")
    print(f"{'='*80}")
    print(f"  Last timestamp: {last_ts}")
    print(f"  Last position:  {last_row['Latitude']:.6f}, {last_row['Longitude']:.6f}")
    print(f"  Last speed:     {last_row['speed_kmh']:.1f} km/h")
    print(f"  Last COG:       {last_row['COG']:.1f}°")
    
    # Get road position for each beyond timestamp
    beyond_points = []
    for sec in range(1, n_beyond + 1):
        new_ts = last_ts + pd.Timedelta(seconds=sec)
        road_pos = get_road_position(str(new_ts), last_row, net)
        if road_pos:
            beyond_points.append(road_pos)
    
    if not beyond_points:
        print("  Could not generate road positions")
        return None
    
    print(f"\n  Expected road positions:")
    for bp in beyond_points:
        print(f"    +{bp['time_diff']:.0f}s: {bp['lat']:.6f}, {bp['lon']:.6f} | "
              f"{bp['edge']} | {bp['dist_from_car']}m")
    
    # Generate with model at each beyond timestamp
    
    results = []
    
    for bp in beyond_points:
        ts_str = bp['timestamp']
        
        
        # Mode 2: timestamp + GPS from road
        gen_m1 = generate_fn(device_id, ts_str, n_steps=1, mode='with_gps',
                             lat=bp['lat'], lon=bp['lon'])
        
        row_result = {'road': bp, 'm1': None}
        
        
        # Print Mode 2
        if gen_m1 is not None and len(gen_m1) > 0:
            m1 = gen_m1.iloc[0]
            m1v = validate_point(m1['Latitude'], m1['Longitude'], m1['speed_kmh'])
            row_result['m1'] = m1.to_dict()
            row_result['m1']['validation'] = m1v
            

        
        print()
        results.append(row_result)
    
    # Consistency check vs last real row
    print(f"\n{'='*80}")
    print(f"  CONSISTENCY: Last Real vs Generated")
    print(f"{'='*80}")
    
    m1_first = results[0]['m1'] if results[0]['m1'] else None
    m1_last = results[-1]['m1'] if results[-1]['m1'] else None
    
    if m1_first and m1_last:
        features = [
            ('speed_kmh', 'Speed (km/h)'),
            ('PCell_RSRP_max', 'RSRP (dBm)'),
            ('PCell_RSRQ_max', 'RSRQ (dBm)'),
            ('PCell_RSSI_max', 'RSSI (dBm)'),
            ('PCell_SNR_1', 'SNR 1 (dB)'),
            ('PCell_SNR_2', 'SNR 2 (dB)'),
            ('PCell_freq_MHz', 'Frequency (MHz)'),
            ('PCell_Downlink_frequency', 'DL Frequency'),
            ('PCell_Band_Indicator', 'Band Indicator'),
            ('PCell_Downlink_bandwidth_MHz', 'DL BW (MHz)'),
            ('datarate', 'Datarate (log)'),
            ('ping_ms', 'Ping (log)'),
            ('jitter_log', 'Jitter (log)'),
            ('temperature', 'Temperature'),
        ]
        
        print(f"\n{'Feature':35s} | {'Last Real':>12s} | {'M1 +1s':>12s} | "
              f"{'M1 +{0}s'.format(n_beyond):>12s} | {'Diff +1s':>10s} | "
              f"{'Diff +{0}s'.format(n_beyond):>10s}")
        print("─" * 95)
        
        for col, label in features:
            if col in last_row.index and col in m1_first:
                real_val = float(last_row[col])
                val_1 = float(m1_first[col])
                val_n = float(m1_last[col])
                diff_1 = val_1 - real_val
                diff_n = val_n - real_val
                
                print(f"{label:35s} | {real_val:12.2f} | {val_1:12.2f} | "
                      f"{val_n:12.2f} | {diff_1:+10.2f} | {diff_n:+10.2f}")

  
    
   
    # Last 5 real rows
    last_5 = segment.tail(n_beyond).copy()
    last_5['source'] = 'REAL'
    
    # Generated rows
    gen_rows = []
    for entry in results:
        if entry['m1']:
            row = entry['m1'].copy()
            if 'validation' in row:
                row.pop('validation')
            row['source'] = 'GENERATED'
            gen_rows.append(row)
    
    gen_df = pd.DataFrame(gen_rows)
    
    # Use only columns that exist in both
    common_cols = ['source'] + [c for c in last_5.columns if c in gen_df.columns and c != 'source']
    
    combined = pd.concat([
        last_5[common_cols].reset_index(drop=True),
        gen_df[common_cols].reset_index(drop=True)
    ], ignore_index=True)
    
    # Fix timestamp
    combined['timestamp'] = combined['timestamp'].apply(
        lambda x: str(x).split('+')[0] if '+' in str(x) else str(x)
    )
    
    # Save
    excel_path = os.path.join(SUMO_DIR, f'beyond_validation_{model_name.replace(" ", "_")}_{device_id}.xlsx')
    combined.to_excel(excel_path, index=False, sheet_name='Beyond Validation')
    
    print(f"\n  Excel saved: {excel_path}")
    print(f"  Rows: {len(last_5)} real + {len(gen_rows)} generated = {len(combined)} total")
    print(f"  Columns: {len(combined.columns)}")
    
    return results
    

In [45]:
# First load dataset and setup
df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')
df = df.sort_values('timestamp').reset_index(drop=True)
df['sin_COG'] = np.sin(np.radians(df['COG']))
df['cos_COG'] = np.cos(np.radians(df['COG']))
df['jitter_log'] = np.log1p(df['jitter'])

pc1 = df[df['device_pc1'] == 1].sort_values('timestamp').reset_index(drop=True)



<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Aditya\AppData\Local\Temp\ipykernel_14648\2799653384.py:2: SyntaxWarning: invalid escape sequence '\m'
  df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')


In [46]:
#pip install pyproj

In [47]:
# Get all unique dates
dates = sorted(pc1['timestamp'].dt.date.unique())
print(f"Available dates: {dates}")

# Day 1
day1 = pc1[pc1['timestamp'].dt.date == dates[0]].reset_index(drop=True)

# Day 2
day2 = pc1[pc1['timestamp'].dt.date == dates[1]].reset_index(drop=True)

# Day 3
day3 = pc1[pc1['timestamp'].dt.date == dates[2]].reset_index(drop=True)

Available dates: [datetime.date(2021, 6, 22), datetime.date(2021, 6, 23), datetime.date(2021, 6, 24)]


In [48]:
# ============================================================
# 1. get_road_position — needs a timestamp and last_row
# ============================================================
print("="*60)
print("1. get_road_position")
print("="*60)

# last_row = any row from the dataset (dict-like with Latitude, Longitude, speed_kmh, COG, timestamp)
last_row = pc1.iloc[-1]  # last row of pc1
print(f"Last row: {last_row['timestamp']}, {last_row['Latitude']:.6f}, {last_row['Longitude']:.6f}, speed={last_row['speed_kmh']:.1f}")



1. get_road_position
Last row: 2021-06-24 18:59:59+02:00, 52.513890, 13.335033, speed=0.0


In [49]:
# timestamp = any future timestamp (string)
pos = get_road_position('2021-06-24 19:30:02+02:00', last_row)
print(f"Result: {pos}")

C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")


Result: {'lat': 52.514848828764045, 'lon': 13.356417774992915, 'edge': 'Straße des 17. Juni', 'speed_limit': 50.004000000000005, 'dist_from_car': 1803.0, 'time_diff': 1803.0, 'timestamp': '2021-06-24 19:30:02+02:00'}


In [50]:
# ============================================================
# 2. validate_point — needs lat, lon, speed
# ============================================================
print("\n" + "="*60)
print("2. validate_point")
print("="*60)

result = validate_point(52.5148, 13.3564, 52.0)
print(f"Result: {result}")



2. validate_point
Result: {'on_road': True, 'snap_dist': 5.3, 'speed_ok': True, 'speed_limit': 50.0, 'road': 'Straße des 17. Juni'}


In [51]:
#pip install openpyxl

In [52]:
gen1=generate_chain('pc1', '2021-06-24 19:30:00+02:00', n_steps=5, mode='with_gps', lat=52.5148, lon=13.3564)
gen1

,timestamp,hour,day_of_week,device_pc1,device_pc2,device_pc3,device_pc4,direction_uplink,measured_qos_delay,measurement,...,PCell_Uplink_Num_RBs,PCell_Uplink_TB_Size,PCell_Uplink_Tx_Power_(dBm),datarate,jitter_log,Pos in Ref Round,target_datarate,ping_ms,COG,jitter
0,2021-06-24 19:30:00+02:00,19,4,1,0,0,0,0,0,16,...,6.263404,9.200404,4.724867,17.996641,-0.003909,8.789499,18.986649,7.060613,81.848602,-0.003901
1,2021-06-24 19:30:01+02:00,19,4,1,0,0,0,0,0,16,...,6.593619,9.795484,4.752413,17.942307,-0.003734,8.765227,18.936354,7.230232,81.672089,-0.003727
2,2021-06-24 19:30:02+02:00,19,4,1,0,0,0,0,0,16,...,6.839551,10.196674,4.763742,17.888987,-0.003408,8.766749,18.890120,7.308496,81.538574,-0.003402
3,2021-06-24 19:30:03+02:00,19,4,1,0,0,0,0,0,16,...,7.058645,10.526399,4.770441,17.825748,-0.003132,8.780957,18.830708,7.356945,81.604843,-0.003127
4,2021-06-24 19:30:04+02:00,19,4,1,0,0,0,0,0,16,...,7.218139,10.771373,4.779252,17.760197,-0.002862,8.809678,18.763828,7.370217,81.800652,-0.002858


In [ ]:
formatted = format_output(gen1)
formatted

,timestamp,device,direction,measured_qos,Latitude,Longitude,speed_kmh,COG,Altitude,precipIntensity,...,PCell_Uplink_TB_Size,PCell_DL_RBs_MCS_Low,PCell_DL_RBs_MCS_Mid,PCell_DL_RBs_MCS_High,PCell_Downlink_Num_RBs,PCell_Uplink_Num_RBs,ping_ms,Pos in Ref Round,Traffic Distance,PCell_Uplink_Tx_Power_(dBm)
0,2021-06-24 19:30:00+02:00,pc1,downlink,datarate,52.514800,13.356400,0.108430,81.848602,26.897148,0.149661,...,9900.129883,127.524811,4.471327,87834.890625,86348.187500,524.002930,1164.158813,6563.944336,26.693527,111.715546
1,2021-06-24 19:30:01+02:00,pc1,downlink,datarate,52.514849,13.356421,0.121416,81.672089,26.792763,0.146287,...,17951.480469,155.963394,10.284625,84104.929688,85952.179688,729.419373,1379.543091,6406.518555,7.829017,114.863503
2,2021-06-24 19:30:02+02:00,pc1,downlink,datarate,52.514850,13.356436,0.121989,81.538574,26.620899,0.141231,...,26812.863281,172.757477,25.356627,81754.250000,85313.140625,933.070068,1491.930786,6416.278320,2.472721,116.183662
3,2021-06-24 19:30:03+02:00,pc1,downlink,datarate,52.514851,13.356451,0.153585,81.604843,26.384781,0.137004,...,37285.949219,188.581436,40.787476,82871.609375,87875.609375,1161.868652,1566.041992,6508.104980,1.714546,116.971260
4,2021-06-24 19:30:04+02:00,pc1,downlink,datarate,52.514852,13.356465,0.171553,81.800652,26.194607,0.134153,...,47636.367188,208.349899,74.109344,79708.820312,88086.968750,1362.947998,1586.978149,6697.762207,1.343694,118.015297


In [55]:

# ============================================================
# 3. validate_beyond — needs a segment DataFrame, generate function, model name
# ============================================================
print("\n" + "="*60)
print("3. validate_beyond")
print("="*60)

# segment = a continuous driving segment ending with car MOVING
# Find Day 1 last segment where car is moving

# Get all unique dates

gaps = day1['timestamp'].diff().dt.total_seconds()
break_points = gaps[gaps > 60].index.tolist()
starts = [0] + break_points
ends = break_points + [len(day1)]

# Pick last segment with moving car
segment = None
for s, e in reversed(list(zip(starts, ends))):
    seg = day1.iloc[s:e]
    if seg['speed_kmh'].iloc[-5:].mean() > 5 and len(seg) > 100:
        segment = seg.reset_index(drop=True)
        break

if segment is None:
    # Fallback: use last segment
    segment = day1.iloc[starts[-1]:ends[-1]].reset_index(drop=True)

print(f"Segment: {len(segment)} rows, {segment['timestamp'].iloc[0]} to {segment['timestamp'].iloc[-1]}")
print(f"End speed: {segment['speed_kmh'].iloc[-1]:.1f} km/h")

results = validate_beyond(segment, generate_chain, 'GRU chain', n_beyond=5)


3. validate_beyond
Segment: 2942 rows, 2021-06-22 17:24:58+02:00 to 2021-06-22 18:13:59+02:00
End speed: 2.4 km/h

  BEYOND-ENDPOINT VALIDATION: GRU chain
  Last timestamp: 2021-06-22 18:13:59+02:00
  Last position:  52.513543, 13.335668
  Last speed:     2.4 km/h
  Last COG:       23.0°


C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")



  Expected road positions:
    +1s: 52.513579, 13.335545 | Straße des 17. Juni | 1.0m
    +2s: 52.513578, 13.335530 | Straße des 17. Juni | 2.0m
    +3s: 52.513577, 13.335515 | Straße des 17. Juni | 3.0m
    +4s: 52.513576, 13.335501 | Straße des 17. Juni | 4.0m
    +5s: 52.513575, 13.335486 | Straße des 17. Juni | 5.0m


C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")


C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")


C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")


C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")


C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")




  CONSISTENCY: Last Real vs Generated

Feature                             |    Last Real |       M1 +1s |       M1 +5s |   Diff +1s |   Diff +5s
───────────────────────────────────────────────────────────────────────────────────────────────
Speed (km/h)                        |         2.43 |         2.49 |         2.49 |      +0.06 |      +0.06
RSRP (dBm)                          |       -86.79 |       -86.46 |       -86.46 |      +0.33 |      +0.33
RSRQ (dBm)                          |        -9.45 |        -9.66 |        -9.66 |      -0.22 |      -0.22
RSSI (dBm)                          |       -57.96 |       -57.59 |       -57.59 |      +0.38 |      +0.38
SNR 1 (dB)                          |        16.83 |        16.40 |        16.40 |      -0.43 |      -0.43
SNR 2 (dB)                          |        18.73 |        16.91 |        16.91 |      -1.82 |      -1.82
Frequency (MHz)                     |      1800.00 |      1800.00 |      1800.00 |      +0.00 |      +0.00
DL Freq

In [56]:
def create_sumo_gui_beyond(segment, beyond_results, device_id='pc1'):
    device_data = df[df[f'device_{device_id}'] == 1].sort_values('timestamp').reset_index(drop=True)
    
    net_dir = os.path.dirname(net_file)
    add_file = os.path.join(SUMO_DIR, f'beyond_{device_id}.add.xml')
    route_file = os.path.join(net_dir, f'{device_id}.rou.xml')
    
    with open(add_file, 'w') as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write('<additional>\n')
        
        sampled = device_data.iloc[::5].reset_index(drop=True)
        print(f"Snapping {len(sampled)} GPS points to roads...")
        
        coords = []
        edge_ids = []
        for _, row in sampled.iterrows():
            x, y = net.convertLonLat2XY(row['Longitude'], row['Latitude'])
            nearby = net.getNeighboringEdges(x, y, r=50)
            if nearby:
                valid = [(e, d) for e, d in nearby if not e.getID().startswith(':')]
                if not valid:
                    valid = nearby
                edge, dist = sorted(valid, key=lambda e: e[1])[0]
                shape = edge.getShape()
                min_d = float('inf')
                snap_x, snap_y = x, y
                for i in range(len(shape) - 1):
                    x1, y1 = shape[i]
                    x2, y2 = shape[i + 1]
                    dx, dy = x2 - x1, y2 - y1
                    seg_len = np.sqrt(dx**2 + dy**2)
                    if seg_len > 0:
                        t = max(0, min(1, ((x - x1)*dx + (y - y1)*dy) / seg_len**2))
                        px, py = x1 + t*dx, y1 + t*dy
                        d = np.sqrt((x - px)**2 + (y - py)**2)
                        if d < min_d:
                            min_d = d
                            snap_x, snap_y = px, py
                coords.append(f"{snap_x:.2f},{snap_y:.2f}")
                edge_ids.append(edge.getID())
            else:
                coords.append(f"{x:.2f},{y:.2f}")
                edge_ids.append(None)
        
        gaps = device_data.iloc[::5]['timestamp'].diff().dt.total_seconds()
        break_indices = [0]
        for i, g in enumerate(gaps):
            if g and g > 120:
                break_indices.append(i)
        break_indices.append(len(coords))
        
        seg_count = 0
        for si in range(len(break_indices) - 1):
            start = break_indices[si]
            end = break_indices[si + 1]
            seg_coords = coords[start:end]
            if len(seg_coords) >= 2:
                f.write(f'    <poly id="route_seg{seg_count}" '
                        f'type="GPS trace segment {seg_count}" '
                        f'color="30,100,255,180" fill="0" layer="5" lineWidth="4" '
                        f'shape="{" ".join(seg_coords)}"/>\n')
                seg_count += 1
        
        print(f"Created {seg_count} trace segments")
        
        last_row = segment.iloc[-1]
        lx, ly = net.convertLonLat2XY(last_row['Longitude'], last_row['Latitude'])
        f.write(f'    <poi id="LAST_REAL" x="{lx:.2f}" y="{ly:.2f}" '
                f'color="255,255,255,255" '
                f'type="LAST REAL | {last_row["timestamp"]} | '
                f'Speed:{last_row["speed_kmh"]:.1f}kmh | '
                f'RSRP:{last_row["PCell_RSRP_max"]:.1f}dBm | '
                f'Freq:{last_row["PCell_freq_MHz"]:.0f}MHz" '
                f'layer="20"/>\n')
        
        line_coords = [f"{lx:.2f},{ly:.2f}"]
        
        for i, entry in enumerate(beyond_results):
            road = entry['road']
            gen = entry['m1']
            
            bx, by = net.convertLonLat2XY(road['lon'], road['lat'])
            line_coords.append(f"{bx:.2f},{by:.2f}")
            
            if gen:
                f.write(f'    <poi id="GEN_+{i+1}s" x="{bx:.2f}" y="{by:.2f}" '
                        f'color="255,140,0,255" '
                        f'type="GEN +{i+1}s | '
                        f'Speed:{gen["speed_kmh"]:.1f}kmh | '
                        f'RSRP:{gen["PCell_RSRP_max"]:.1f}dBm | '
                        f'Freq:{gen["PCell_freq_MHz"]:.0f}MHz | '
                        f'DR:{gen["datarate"]:.2f} | '
                        f'Ping:{gen["ping_ms"]:.2f}" '
                        f'layer="21"/>\n')
            else:
                f.write(f'    <poi id="ROAD_+{i+1}s" x="{bx:.2f}" y="{by:.2f}" '
                        f'color="255,165,0,255" '
                        f'type="ROAD +{i+1}s | {road["edge"]}" '
                        f'layer="21"/>\n')
        
        f.write(f'    <poly id="beyond_path" type="Generated beyond path" '
                f'color="255,140,0,200" fill="0" layer="6" lineWidth="5" '
                f'shape="{" ".join(line_coords)}"/>\n')
        
        f.write('</additional>\n')

    # --- Generate route file ---
    valid_edges = [eid for eid in edge_ids if eid is not None]
    # Deduplicate consecutive duplicate edges
    deduped_edges = [valid_edges[0]] if valid_edges else []
    for eid in valid_edges[1:]:
        if eid != deduped_edges[-1]:
            deduped_edges.append(eid)

    print(f"Writing route file with {len(deduped_edges)} edges...")
    with open(route_file, 'w') as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write('<routes>\n')
        f.write(f'    <vType id="{device_id}_type" vClass="passenger"/>\n')
        f.write(f'    <route id="{device_id}_route" edges="{" ".join(deduped_edges)}"/>\n')
        f.write(f'    <vehicle id="{device_id}" type="{device_id}_type" '
                f'route="{device_id}_route" depart="0"/>\n')
        f.write('</routes>\n')
    print(f"Route file written: {route_file}")

    # --- Generate SUMO config ---
    cfg_file = os.path.join(SUMO_DIR, f'beyond_{device_id}.sumocfg')
    
    # --- Generate SUMO config ---
    with open(cfg_file, 'w') as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write('<configuration>\n')
        f.write('    <input>\n')
        f.write(f'        <net-file value="{net_file}"/>\n')
        f.write(f'        <route-files value="{route_file}"/>\n')
        f.write(f'        <additional-files value="{add_file}"/>\n')
        f.write('    </input>\n')
        f.write('</configuration>\n')
    
    print(f"\n{'='*60}")
    print(f"SUMO GUI ready:")
    print(f"~/sumo/bin/sumo-gui -c {cfg_file}")
    print(f"{'='*60}")
    print(f"\nBlue lines: full {device_id} route (snapped to roads)")
    print(f"White dot:  last real position")
    print(f"Orange line + dots: generated beyond points")
    print(f"Locate → POI → LAST_REAL to find beyond points")
    print(f"\nFiles created:")
    print(f"  {add_file}")
    print(f"  {route_file}")
    print(f"  {cfg_file}")
    print(f"\nFor Windows: copy both files + {net_file} to your Windows machine")
    print(f"Then run: sumo-gui -c {os.path.basename(cfg_file)}")

In [57]:
# Then create SUMO GUI
create_sumo_gui_beyond(segment, results, device_id='pc1')

Snapping 11904 GPS points to roads...


C:\Program Files (x86)\Eclipse\Sumo\tools\sumolib\net\__init__.py:370: UserWarning: Module 'rtree' not available. Using brute-force fallback.
  warnings.warn("Module 'rtree' not available. Using brute-force fallback.")


Created 19 trace segments
Writing route file with 2711 edges...
Route file written: D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\pc1.rou.xml

SUMO GUI ready:
~/sumo/bin/sumo-gui -c D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\beyond_pc1.sumocfg

Blue lines: full pc1 route (snapped to roads)
White dot:  last real position
Orange line + dots: generated beyond points
Locate → POI → LAST_REAL to find beyond points

Files created:
  D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\beyond_pc1.add.xml
  D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\pc1.rou.xml
  D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\beyond_pc1.sumocfg

For Windows: copy both files + D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\osm1.net.xml to your Windows machine
Then run: sumo-gui -c beyond_pc1.sumocfg
